# So sánh hiệu năng và chất lượng 5 Pipeline OCR trên GPU

Notebook này triển khai và đánh giá 5 phương pháp OCR khác nhau:
1. **PaddleOCR + VietOCR** (Nhận diện tiếng Việt chất lượng cao)
2. **PaddleOCR Only** (End-to-end cả phát hiện và nhận diện bằng PaddleOCR)
3. **Florence-2** (Vision-Language Model đa năng của Microsoft)
4. **PaddleOCR + TrOCR** (Dùng PaddleOCR phát hiện vùng văn bản và TrOCR của Microsoft nhận diện)
5. **EasyOCR** (Cả phát hiện và nhận diện bằng EasyOCR)

In [ ]:
import os
import gc
import cv2
import time
import json
import torch
import paddle
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# --- Vá lỗi Pillow ANTIALIAS ---
if not hasattr(Image, "ANTIALIAS"):
    Image.ANTIALIAS = Image.Resampling.LANCZOS

# --- Vá lỗi tương thích cho transformers v5.x khi chạy Florence-2 ---
import transformers
transformers.configuration_utils.PretrainedConfig.forced_bos_token_id = None
transformers.tokenization_utils_base.PreTrainedTokenizerBase.additional_special_tokens = property(
    lambda self: getattr(self, 'extra_special_tokens', [])
)
from transformers.cache_utils import EncoderDecoderCache
EncoderDecoderCache.__getitem__ = lambda self, idx: (
    self.self_attention_cache.layers[idx].keys,
    self.self_attention_cache.layers[idx].values,
    self.cross_attention_cache.layers[idx].keys,
    self.cross_attention_cache.layers[idx].values,
)
EncoderDecoderCache.__len__ = lambda self: len(self.self_attention_cache.layers)
# -----------------------------------------------------------------

# Kiểm tra GPU hoạt động cho PyTorch & Paddle
print("--- KIỂM TRA THIẾT BỊ GPU ---")
print("PyTorch CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("PyTorch GPU Device:", torch.cuda.get_device_name(0))

print("Paddle compiled with CUDA:", paddle.is_compiled_with_cuda())
paddle.device.set_device('gpu')


In [ ]:
ROOT = Path(".")
FRAME_PATH = Path("..") / "data" / "frame"
COMPARISON_OUT = ROOT / "ocr_comparison_results.json"
VISUAL_PATH = ROOT / "visualize"

In [ ]:
def get_rotated_crop_image(img, points):
    """
    Cắt ảnh dựa trên 4 điểm tọa độ từ PaddleOCR. Xử lý cả box bị xoay/nghiêng.
    """
    points = np.array(points, dtype=np.float32)
    rect = cv2.minAreaRect(points)
    box = cv2.boxPoints(rect)
    box = box.astype(np.int32)

    width = int(rect[1][0])
    height = int(rect[1][1])

    src_pts = points.astype("float32")
    dst_pts = np.array([[0, 0],
                        [width - 1, 0],
                        [width - 1, height - 1],
                        [0, height - 1]], dtype="float32")

    M = cv2.getPerspectiveTransform(src_pts, dst_pts)
    warped = cv2.warpPerspective(img, M, (width, height))
    
    if height > width * 1.2:
        warped = cv2.rotate(warped, cv2.ROTATE_90_CLOCKWISE)
        
    return warped


### Khởi tạo các Mô hình (Tất cả chạy trên GPU)

In [ ]:
from paddleocr import PaddleOCR
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg
from transformers import AutoModelForCausalLM, AutoProcessor, TrOCRProcessor, VisionEncoderDecoderModel
import easyocr

print("--- Khởi tạo 1. PaddleOCR & VietOCR ---")
det_model = PaddleOCR(use_angle_cls=False, lang='vi')

config = Cfg.load_config_from_name('vgg_seq2seq')
config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
vietocr_predictor = Predictor(config)

print("--- Khởi tạo 2. Florence-2 (base) ---")
florence_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Florence-2-base", 
    trust_remote_code=True,
    attn_implementation="eager"
).eval().to("cuda")
florence_processor = AutoProcessor.from_pretrained(
    "microsoft/Florence-2-base", 
    trust_remote_code=True
)

# --- Vá lỗi cache rỗng (prefill) cho Florence-2 language model ---
original_prepare = florence_model.language_model.prepare_inputs_for_generation
original_forward = florence_model.language_model.forward
clean_past = lambda past: None if (
    past is not None and 
    hasattr(past, 'self_attention_cache') and 
    (len(past.self_attention_cache.layers) == 0 or getattr(past.self_attention_cache.layers[0], 'keys', None) is None)
) else past

florence_model.language_model.prepare_inputs_for_generation = lambda input_ids, past_key_values=None, **kwargs: original_prepare(
    input_ids, past_key_values=clean_past(past_key_values), **kwargs
)
florence_model.language_model.forward = lambda *args, **kwargs: original_forward(
    *args, **{**kwargs, 'past_key_values': clean_past(kwargs.get('past_key_values'))} if 'past_key_values' in kwargs else kwargs
)
# ---------------------------------------------------------------

print("--- Khởi tạo 3. TrOCR (base-printed) ---")
trocr_processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)
trocr_model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
).eval().to("cuda")

print("--- Khởi tạo 4. EasyOCR ---")
easyocr_reader = easyocr.Reader(['vi', 'en'], gpu=True)

print("========================================\nKhởi tạo tất cả các mô hình hoàn tất!")


### Định nghĩa Hàm chạy cho từng Pipeline

In [ ]:
# 1. PaddleOCR + VietOCR
def run_paddle_vietocr(image_path, img):
    result = det_model.ocr(img)
    if not result or not result[0]:
        return ""
    first_result = result[0]
    boxes = first_result.get('dt_polys', []) if isinstance(first_result, dict) else first_result
    full_text_list = []
    for points in boxes:
        cropped_img_cv = get_rotated_crop_image(img, points)
        cropped_img_rgb = cv2.cvtColor(cropped_img_cv, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(cropped_img_rgb)
        try:
            text = vietocr_predictor.predict(pil_img)
            if text.strip():
                full_text_list.append(text)
        except Exception as e:
            pass
    return " ".join(full_text_list)

# 2. PaddleOCR only
def run_paddle_only(image_path, img):
    result = det_model.ocr(img)
    if not result or not result[0]:
        return ""
    first_result = result[0]
    rec_texts = first_result.get('rec_texts', []) if isinstance(first_result, dict) else [line[1][0] for line in first_result]
    return " ".join([t for t in rec_texts if t.strip()])

# 3. Florence-2
def run_florence2(image_path, img):
    # Tác vụ yêu cầu ảnh vuông để tránh lỗi assertion trong vision tower của Florence-2
    image = Image.open(image_path).convert("RGB").resize((768, 768))
    inputs = florence_processor(text="<OCR>", images=image, return_tensors="pt")
    # Đưa inputs lên cuda và ép kiểu sang float16 khớp với model weights trên GPU
    inputs = {k: v.to('cuda', torch.float16) if v.dtype == torch.float32 else v.to('cuda') for k, v in inputs.items()}
    generated_ids = florence_model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3
    )
    generated_text = florence_processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed_answer = florence_processor.post_process_generation(
        generated_text,
        task="<OCR>",
        image_size=(image.width, image.height)
    )
    return parsed_answer["<OCR>"].strip()

# 4. PaddleOCR + TrOCR
def run_paddle_trocr(image_path, img):
    result = det_model.ocr(img)
    if not result or not result[0]:
        return ""
    first_result = result[0]
    boxes = first_result.get('dt_polys', []) if isinstance(first_result, dict) else first_result
    full_text_list = []
    for points in boxes:
        cropped_img_cv = get_rotated_crop_image(img, points)
        cropped_img_rgb = cv2.cvtColor(cropped_img_cv, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(cropped_img_rgb)
        try:
            pixel_values = trocr_processor(pil_img, return_tensors="pt").pixel_values.to("cuda")
            generated_ids = trocr_model.generate(pixel_values)
            text = trocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            if text.strip():
                full_text_list.append(text)
        except Exception as e:
            pass
    return " ".join(full_text_list)

# 5. EasyOCR
def run_easyocr(image_path, img):
    results = easyocr_reader.readtext(img, detail=0)
    return " ".join(results)


### Đánh giá và Đo thời gian chạy

In [ ]:
folders = os.listdir(FRAME_PATH)
image_files = []
for folder in folders:
    folder_path = FRAME_PATH / folder
    if folder_path.is_dir():
        for f in sorted(os.listdir(folder_path)):
            if f.lower().endswith('.webp'):
                image_files.append((folder, f, folder_path / f))

print(f"Tìm thấy {len(image_files)} ảnh để so sánh.")

pipelines = {
    "PaddleOCR+VietOCR": run_paddle_vietocr,
    "PaddleOCR Only": run_paddle_only,
    "Florence-2": run_florence2,
    "PaddleOCR+TrOCR": run_paddle_trocr,
    "EasyOCR": run_easyocr
}

results_log = []

for folder, file_name, path in tqdm(image_files, desc="Đang xử lý các pipeline"):
    img = cv2.imread(str(path))
    if img is None:
        continue
    
    frame_idx = file_name.removeprefix('keyframe_').removesuffix('.webp')
    key = f"{folder}_{frame_idx}"
    entry = {"image_key": key}
    
    for pipe_name, run_fn in pipelines.items():
        torch.cuda.empty_cache()
        gc.collect()
        
        # Đo thời gian
        start = time.time()
        try:
            text = run_fn(str(path), img)
        except Exception as e:
            text = f"[Lỗi: {str(e)}]"
        duration = time.time() - start
        
        entry[f"{pipe_name}_text"] = text
        entry[f"{pipe_name}_time"] = duration
        
    results_log.append(entry)

# Lưu kết quả dạng JSON
with open(COMPARISON_OUT, "w", encoding="utf-8") as f:
    json.dump(results_log, f, ensure_ascii=False, indent=4)

print(f"Đã lưu kết quả so sánh tại: {COMPARISON_OUT}")


### Kết quả văn bản nhận diện

In [ ]:
df = pd.DataFrame(results_log)
text_cols = ["image_key"] + [f"{name}_text" for name in pipelines.keys()]
df_text = df[text_cols]
# Hiển thị toàn bộ bảng kết quả dạng văn bản
df_text

### Kết quả thời gian thực thi (Latency)

In [ ]:
time_cols = ["image_key"] + [f"{name}_time" for name in pipelines.keys()]
df_time = df[time_cols]
print("Thời gian xử lý trung bình mỗi ảnh (giây):")
avg_times = df_time.mean(numeric_only=True)
for name, val in avg_times.items():
    pipe_name = name.removesuffix("_time")
    print(f"- {pipe_name}: {val:.4f} s")
